In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)

In [ ]:
data_cd = 'data/EAE/attn'
mdata = mu.read(data_cd+"/EAE_HV_then_merged_test293883_atchleyEmbs_allcells.h5mu")
mdata

In [3]:
tcr_embs = pd.DataFrame(mdata.obsm['tcr_embs'], index=mdata.obs_names)
tcr_embs.head(5)

,0,1,2,3,4,5,6,7,8,9,...,476,477,478,479,480,481,482,483,484,485
AAGTAGCAGATAGGCG-1_0516_CNS,0.0,0.0,0.326822,-0.363999,0.00776,0.338771,-0.096833,0.231076,0.398391,-0.439795,...,-0.030074,-0.096837,-0.015646,-0.069109,-0.081029,-0.115975,0.0,-0.583787,1.110482,-0.285230
AAGTATACACCCAGTA-1_0516_CNS,0.0,0.0,0.326822,-0.363999,0.00776,0.338771,-0.096833,0.231076,0.398391,-0.439795,...,-0.030074,-0.096837,-0.015646,-0.069109,-0.081029,-0.115975,0.0,0.818493,0.346619,-0.285230
AAGTTTGGTGGAACCC-1_0516_CNS,0.0,0.0,0.326822,-0.363999,0.00776,0.338771,-0.096833,0.231076,0.398391,-0.439795,...,-0.030074,-0.096837,-0.015646,-0.069109,-0.081029,-0.115975,0.0,1.519633,0.346619,-0.285230
AATGCGACACTAAAGG-1_0516_CNS,0.0,0.0,0.326822,-0.363999,0.00776,0.338771,-0.096833,0.231076,0.398391,-0.439795,...,-0.030074,-0.096837,-0.015646,-0.069109,-0.081029,-0.115975,0.0,-0.583787,-1.944973,-0.285230
AATGCGACACTATGAC-1_0516_CNS,0.0,0.0,0.326822,-0.363999,0.00776,0.338771,-0.096833,0.231076,0.398391,-0.439795,...,-0.030074,-0.096837,-0.015646,-0.069109,-0.081029,-0.115975,0.0,-1.986068,-0.417245,-0.264592


In [4]:
gex_df = mdata['gex'].to_df()
gex_df = gex_df.loc[:, ~gex_df.columns.str.startswith('mt-')]
gex_df.head(5)

,1110034G24Rik,1110037F02Rik,1500009L16Rik,1700003C15Rik,1700012B07Rik,1700016L21Rik,1700019D03Rik,1700025G04Rik,1700028E10Rik,1700061F12Rik,...,Zfp831,Zfyve28,Zg16,Zhx2,Zmym2,Zmym4,Znrf3,Zswim6,Zup1,Zzz3
AAGTAGCAGATAGGCG-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0
AAGTATACACCCAGTA-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,2.22883,0.000000,0.0
AAGTTTGGTGGAACCC-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0
AATGCGACACTAAAGG-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.992625,0.992625,0.0,0.00000,0.992625,0.0
AATGCGACACTATGAC-1_0516_CNS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0


In [6]:
labels = mdata['gex'].obs[['tissue', 'cell_type', 'state', 'GSE', 'sample_id']]
labels = pd.concat([labels, mdata.obs['set']], axis=1)
labels = pd.concat([labels, mdata['airr'].obs['clone_id_size']], axis=1)
labels.columns = [f"label_{col}" for col in labels.columns]

labels.head(5)

,label_tissue,label_cell_type,label_state,label_GSE,label_sample_id,label_set,label_clone_id_size
AAGTAGCAGATAGGCG-1_0516_CNS,CNS,CD4,Activation,LEE,5_7,train,1.0
AAGTATACACCCAGTA-1_0516_CNS,CNS,NaN,Activation,LEE,5_7,train,1.0
AAGTTTGGTGGAACCC-1_0516_CNS,CNS,NaN,Exhaust,LEE,5_3,train,1.0
AATGCGACACTAAAGG-1_0516_CNS,CNS,CD8,NaN,LEE,5_7,train,1.0
AATGCGACACTATGAC-1_0516_CNS,CNS,NaN,Activation,LEE,5_3,train,3.0


In [7]:
labels['label_set'].value_counts()

label_set
train    96201
test     10040
Name: count, dtype: int64

### Save Raw gex and tcr

In [ ]:
import os
output_cd = data_cd + '/raw_data'
if not os.path.exists(output_cd):
    os.makedirs(output_cd)
    tcr_embs.to_csv(os.path.join(output_cd, "tcr_embs.csv"), index=True)
    # gex_df.to_csv(os.path.join(output_cd, "gex_df.csv"), index=True)
    labels.to_csv(os.path.join(output_cd, "labels.csv"), index=True)
    
else:
    print(f"Directory {output_cd} already exists")




In [10]:
%pwd

'e:\\Python code\\Machine learning\\JupyterNote\\Bio_CRC\\Data processing'